In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS electronics_retailer_clg.silver;
-- drop TABLE electronics_retailer_clg.silver.exchange_rates

In [0]:
import pyspark.sql.functions as F

data = spark.table("electronics_retailer_clg.bronze.exchange_rates")

def standardize_date(columnName, df):
    return df.withColumn(
        "date_parsed",
        F.coalesce(
            F.try_to_date(F.col(columnName), "M/d/yyyy"),
            F.try_to_date(F.col(columnName), "M-d-yyyy"),
            F.try_to_date(F.col(columnName), "yyyy-M-d"),
            F.try_to_date(F.col(columnName), "yyyy/M/d")
        )
    ).withColumn(
        columnName,
        F.trim(F.col("date_parsed")).cast('date')
    ).drop("date_parsed")

data = standardize_date("date", data)
display(data)

In [0]:
data = data.withColumn("exchange", F.col("exchange").cast("double"))


display(data)
data.printSchema()


data.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("electronics_retailer_clg.silver.exchange_rates")

print("Exchange rates cleaned successfully")